## 데이터 수집 자동화

### 안정적 수집과 저장

#### 수집 오류 유형

- 요소 없음 : CSS 선택지가 틀렸거나 화면에 없음 > 예외처리
- 로딩 지연 : 페이지가 늦게 뜸 > 명시적 대기
- 클릭 실패 : 버튼이 아직 클릭 불가 > 명시적 대기
- 데이터 누락 : 일부 항목이 페이지에 없음 > 기본값 처리
- 중복 수집 : 같은 데이터가 포함 > 판다스 중복처리
- 저장 실패 : 파일 존재여부, 권한여부 > 파일명 변경, 예외처리

In [2]:
try:
    price = input('가격입력 > ')
    print(int(price))
except Exception as e: # 실행중 오류 발생시 처리할 부분
    print(f'가격은 숫자만 입력할 수 있습니다.')
finally: # 실행중 오류가 나든 안 나든 무조건 실행할 부분
    print('프로그램종료')

가격은 숫자만 입력할 수 있습니다.
프로그램종료


In [3]:
from selenium import webdriver

In [4]:
driver = webdriver.Chrome()

try:
    driver.get('https://google.com') # 웹사이트 오픈
    print(driver.title) # 웹사이트 head 영역에 있는 탭이름(title) 가져오기
except Exception as e:
    print(f'실행 중 오류 발생! : {e.args}')
finally:
    driver.quit()

Google


#### find_element 문제점
- find_elements() 실행 중 오류 발생 X
- find_element() 실행중 오류 발생 가능성 높음
    - 함수를 만드는게 유리

In [7]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [5]:
# 안전하게 요소 추출 함수
def get_text_safe(parent, selector, default=''):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text
    except:
        return default # 내가 지정한 기본값이 그냥 리턴

In [8]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)
driver.get('https://www.danawa.com')

# 보통 footer가 가장 아래이니 footer가 다 로딩될 때까지 ~
wait.until(
    EC.visibility_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="41fa6a1ac15a1e6e55f821e4c57428a1", element="f.61D3EF9BE09D4E260B27FFF9BDF21718.d.F540DA2D2A9FBEA7A4E255C99E889DEB.e.3331")>

In [11]:
get_text_safe(driver, 'div.item > a .txt', 'None')

'None'

#### 명시적 대기와 타임 아웃
- 명시적 대기를 하더라도 시간을 초과하면 실행 중 오류가 발생
- 실행 중 오류 잡으려면 trt~except 사용 필요

In [24]:
from selenium.common.exceptions import TimeoutException
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)
try:
    driver.get('https://www.naver.com')
    wait.until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, '.footer'))
    )
    print('로딩 완료!')
except TimeoutException as e:
    print('로딩실패! 타임오버')

로딩실패! 타임오버


#### 버튼 클릭 가능상태까지 대기
- 버튼, 링크가 화면에 나타나도 js가 로딩완료될 떄까지 클릭못하는 경우가 발생할 수 있음
- 클릭상태까지 대기

In [25]:
enable_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#account a'))
)

enable_button.click()

#### 로그파일 저장
- 화면에 출력보다 로그파일 저장이 효율적

In [47]:
import logging

logging.basicConfig(
    filename='scraping.log',
    level=logging.INFO, # DEBUG, INFO, WARN, WARNING, ERROR, FATAL... 낮은 레벨순
    format="%(asctime)s %(levelname)s %(message)s",
    encoding='utf-8',
    force=True
)

In [48]:
logging.info('수집시작')
logging.info('상품데이터 수집완료')
logging.error('수집 중 오류 발생!')

#### DB에 저장

- 이전 저장한 다나와 데이터 사용

In [70]:
df = pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df = df.drop(columns=["Unnamed: 0"], errors='ignore')

In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 897 entries, 0 to 896
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          897 non-null    int64  
 1   computer_name  897 non-null    str    
 2   computer_spec  897 non-null    str    
 3   price_spec     897 non-null    str    
 4   price          897 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.2 KB


#### DB 저장
- SQLAchemy 패키지 사용. SQL 쿼리문 없이 CRUD 가능
- `pip install SQLAlchemy`

In [52]:
!pip install SQLAlchemy

Defaulting to user installation because normal site-packages is not writeable


In [72]:
from datetime import datetime
from getpass import getpass

#SQLAlchemy
from sqlalchemy import create_engine, URL, Text, BigInteger, Integer

In [73]:
df.head()

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1209640.0
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,MSI MAG CODEX Z2 A9NVP MA09 R5-9600X RTX 5070 게이밍,9600X / RTX 5070 / (AMD) B850 / OS미포함 / 750W /...,"16GB, M.2 1TB",2545860.0


In [74]:
# 저장할 테이블명 - danawa_gaming_pc_yymmdd
table_name = 'danawa_gaiming_pc_' + datetime.now().strftime('%y%m%d')
print(table_name)

danawa_gaiming_pc_261006


In [75]:
#PostgreSQL 연결정보
db_url = URL.create(
    'postgresql+psycopg',
    username='postgres',
    password='123456',
    host='127.0.0.1',
    port=5432,
    database='postgres'
)

In [76]:
engine = create_engine(db_url)

In [77]:
# 테이블 생성하고 DataFrame 데이터 저장
df.to_sql(
    name=table_name,
    con=engine,
    if_exists='fail',
    index=False,
    dtype={
        'index' : BigInteger(),
        'computer_name' : Text(),
        'computer_spec' : Text(),
        'price_spec' : Text(),
        'price' : Integer()
    }
)

engine.dispose() # 사용 후 해제

print(f'{len(df)}건을 저장했습니다.')

897건을 저장했습니다.


#### 스케줄링
- 정해진 시간마다 반복 수행하도록 하는 것
- OS 마다 방식이 상이
    - Windows : 작업 스케줄러
    - Linux : cron
    - Python : schedule 라이브러리 가능

##### 윈도우 작업 스케줄러
- 윈도우 > 작업 스케줄러 검색 후 실행
- `기본 작업 만들기...` 클릭
- 기본 작업 만들기 > 이름과 설명(옵션) 추가
- 트리거 탭 > 매일로 선택
    - 작업 내용은 시작시간과 반복 횟수 등 상황에 맞게 변경
- 동작 탭 > 새로 만들기 버튼 클릭
    - 동작 : 프로그램 시작
    - 프로그램 스크립트 : .bat 위치 선택
    - 시작 위치 : D:\SourceBank\ai-datapipeline-2026\chapt04 와 같이 경로 입력
- 속성 창 오픈
    - 보안 옵션 : Administrators(관리자 그룹)을 선택하고 사용자 변경
    - 가장 높은 수준의 권한으로 실행